# Fish Audio S2 Pro - Colab 一键部署

这是一个用于在 Google Colab 上快速运行 `fish-s2-pro-zero` 项目的笔记本。

### 💡 使用说明
1. **启用 GPU**: 点击 `Runtime` -> `Change runtime type` -> 硬件加速器选择 `T4 GPU`。
2. **一键运行**: 点击左上角的 `Runtime` -> `Run all`。
3. **获取链接**: 等待最后一步运行完成后，点击输出的 `public URL` (以 `.gradio.live` 结尾) 即可访问界面。

In [ ]:
# @title 1. 环境准备与仓库拉取
import os

# 仓库地址
REPO_URL = "https://github.com/entishl/fish-s2-pro-colab.git"
REPO_DIR = "fish-s2-pro-colab"

if not os.path.exists(REPO_DIR):
    print(f"正在克隆仓库: {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print("仓库已存在，尝试拉取最新代码...")
    %cd {REPO_DIR}
    !git pull
    %cd ..

%cd {REPO_DIR}

print("正在安装系统依赖...")
!apt-get update && apt-get install -y ffmpeg libsox-dev portaudio19-dev

In [ ]:
# @title 2. 安装 Python 依赖
# 此过程可能需要 3-5 分钟
!pip install -r requirements.txt
!pip install gradio spaces

In [ ]:
# @title 3. 启动 Web 界面
import sys
from unittest.mock import MagicMock

# 模拟 Hugging Face 的 spaces 库
# 因为 Colab 没有 Zero GPU 环境，我们需要拦截 @spaces.GPU 装饰器使其变为普通函数
mock_spaces = MagicMock()
mock_spaces.GPU = lambda **kwargs: (lambda f: f)
sys.modules["spaces"] = mock_spaces

# 清除缓存以防由于 import 导致的逻辑冲突
if 'app' in sys.modules:
    del sys.modules['app']

import app
print("模型正在后台加载，请稍候...")

# 启动 Gradio
# 注意：share=True 是生成公共链接的关键
app.app.launch(share=True, inline=False, debug=True)